# Chuuchuu CSV -> parquet conversion

Run this notebook once per `data_selection`, **before** `Chuuchuu_pipeline.ipynb`. It:

1. Converts the raw CSV extract to a parquet mirror stored next to it on the shared drive (so the pipeline never re-parses a multi-GB CSV).
2. Precomputes `stopName_slug` (used by the pipeline's country-attribution fallback) and bakes it into that same parquet file.

Doing the slugify step here -- in its own short-lived kernel, with nothing else in memory -- avoids the memory pressure it caused when it ran inside the analysis pipeline's already-loaded kernel. `Chuuchuu_pipeline.ipynb` now expects the parquet mirror to already exist and contain `stopName_slug`; if it's missing, that means this notebook hasn't been run yet for this `data_selection`.


In [1]:
import pandas as pd
import os
import pyarrow.parquet as pq


## Step 1 -- Dataset selection & shared drive path resolution

Identical to `Chuuchuu_pipeline.ipynb`'s Step 0 -- keep `data_selection` and the candidate folder list in sync between the two notebooks so they resolve to the same file.


In [2]:
data_selection = "full"  # "full" | "combined" | "june" | "february" | "french" 

# personal PC / VM shared-drive mount points for the Chuuchuu folder -- add a new entry here if
# this pipeline runs somewhere else and the folder isn't found
candidate_chuuchuu_folders = [
    r"G:\.shortcut-targets-by-id\1fop_2EKRGMK369J15pyuC9039CxT_tnw\Rail_databases\punctuality_big_data\Chuuchuu",  # personal PC
    r"D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu",  # VM
]

data_chuuchuu_folder = next((path for path in candidate_chuuchuu_folders if os.path.isdir(path)), None)
if data_chuuchuu_folder is None:
    raise FileNotFoundError(
        "Could not find the Chuuchuu shared-drive folder on this machine -- no path in "
        "candidate_chuuchuu_folders exists. Add this machine's local mount path to the list above."
    )
print(f"Using Chuuchuu folder: {data_chuuchuu_folder}")

Using Chuuchuu folder: D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu


## Step 2 -- Convert to parquet, precomputing `stopName_slug`

Skips the (slow) conversion if an up-to-date mirror already exists -- "up-to-date" means it already carries `stopName_slug`, not just that a file happens to exist at this path. Mirrors built before this notebook existed won't have that column yet and need regenerating once.


In [3]:
raw_csv_filename_by_selection = {
    "combined": "delay_records_test_combined_EU.csv",
    "june": "delay_records_2026-06-10_to_2026-06-17_EU.csv",
    "february": "delay_records_2026-02-26_to_2026-03-04_EU.csv",
    "french": "delay_records_FR_2025.csv",
    "full": "delay_records_full_2025.csv"
}
raw_csv_filename = raw_csv_filename_by_selection[data_selection]
raw_csv_path = f"{data_chuuchuu_folder}/{raw_csv_filename}"
raw_parquet_path = f"{data_chuuchuu_folder}/{os.path.splitext(raw_csv_filename)[0]}.parquet"


def slugify(series):
    return (
        series.str.normalize("NFKD")     # split accented letters from their accents
        .str.encode("ascii", "ignore")   # drop the accents
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.replace(r"\(", "-", regex=True)
        .str.replace(r"\)", "", regex=True)
        .str.replace(r"[^a-z0-9]+", "-", regex=True)  # collapse remaining punctuation/spaces
        .str.strip("-")
    )


needs_conversion = True
if os.path.exists(raw_parquet_path):
    existing_columns = pq.ParquetFile(raw_parquet_path).schema_arrow.names
    if "stopName_slug" in existing_columns:
        needs_conversion = False
        print(f"Parquet mirror already up to date: {raw_parquet_path}")
    else:
        print(f"Parquet mirror at {raw_parquet_path} predates stopName_slug -- regenerating")
else:
    print(f"No parquet mirror yet at {raw_parquet_path} -- converting from CSV (slow, one-off)")

if needs_conversion:
    data_raw = pd.read_csv(raw_csv_path, low_memory=False)
    data_raw["stopName_slug"] = slugify(data_raw["stopName"])
    data_raw.to_parquet(raw_parquet_path)
    print(f"Saved enriched parquet mirror ({data_raw.shape[0]} rows) to {raw_parquet_path}")


Parquet mirror at D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu/delay_records_full_2025.parquet predates stopName_slug -- regenerating
Saved enriched parquet mirror (85236926 rows) to D:\.shortcut-targets-by-id\18oT0tHU4ycBoKA7-PZZc_W0DyRedU_Jt\T&A\Shared knowledge tools and data\data_sources\Rail_databases\punctuality_big_data\Chuuchuu/delay_records_full_2025.parquet
